# v9_b Step 5a: Connect features to network by reaction steps



In [ ]:
import os
import json
from copy import deepcopy
import numpy as np
import pandas as pd
import mass2chem.formula as asari_formula

from mass2chem import formula
from asari.tools import match_features as mf

featdir = 'results/step3_khipus/'
proddir =  'results/step4_predicted/'
outdir = 'results/step5a_f2net/'
os.makedirs(outdir, exist_ok=True)

In [ ]:
# Define the substrates and reagents for the chemical reactions
substrates = {
    'Estradiol': {
        'mz': 506.23596,        # DnCl
            'formula': 'C18H24O2',
            'formula_dict': {'C': 18, 'H': 24, 'O': 2},
            'rtime': 1094, 
            'id': 'Estradiol'
    },
    'Estrone': {
        'mz': 504.22031,
        'formula': 'C18H22O2',
        'formula_dict': {'C': 18, 'H': 22, 'O': 2},
        'rtime': 1099,
        'id': 'Estrone'
    },
    'DHEA': {
        'mz': 536.29414,        # DnHz
        'formula': 'C19H28O2',
        'formula_dict': {'C': 19, 'H': 28, 'O': 2},
        'rtime': 549,
        'id': 'DHEA'
    },
    'Pregnenolone': {
        'mz': 564.32544,
        'formula': 'C21H32O2',
        'formula_dict': {'C': 21, 'H': 32, 'O': 2},
        'rtime': 640,
        'id': 'Pregnenolone'
    },
    'Progesterone': {
        'mz': 562.30979,
        'formula': 'C21H30O2',
        'formula_dict': {'C': 21, 'H': 30, 'O': 2},
        'rtime': 695,
        'id': 'Progesterone'
    }
}

reagents_mz = {
    'DnCl': 233.05105,
    'DnHz': 247.07793,
    'DmPA': 161.08406, 
}

def get_clean_features(list_features):
    '''To separate features of redundant ions. 
    If 'khipu' in feature, collate.
    Return unique features and a dict map of others.
    '''
    new, extra_map, misfits = [], {}, []
    khipus = {}
    for feature in list_features:
        if 'khipu' in feature:
            if feature['khipu'] not in khipus:
                khipus[feature['khipu']] = [feature]
            else:
                khipus[feature['khipu']].append(feature)
            
            if feature['ion_relation'] == 'M0,M+H+':
                feature['is_primary'] = True
                new.append(feature)
        else:
            new.append(feature)
            
    print("Number of khipus:", len(khipus))
    for khipu, features in khipus.items():
        # map redundant features to the representative feature
        _primary = None
        for feature in features:
            if feature.get('is_primary', False):
                _primary = feature['id']
                break
        if not _primary:
            misfits.extend(features)
            print("No primary feature found for khipu:", features)
        else:
            for feature in features:
                if feature['id'] != _primary:
                    extra_map[feature['id']] = _primary
            
    return new, extra_map, misfits

def retrieve_rxn_steps_from_product(input_product):
    '''return [end_product, last step, ... ]
    '''
    product = input_product.copy()
    sequence = [(product['id'], product['mz'], product['formula'], '', ''), ]
    input_cpd = product.get('input_cpd', None)
    while input_cpd:
        rxn = product.get('reaction')
        sequence.append((input_cpd['id'], input_cpd['mz'],
                         input_cpd['formula'], rxn['formula_difference'], rxn['reactionIDs']))
        product = input_cpd
        input_cpd = input_cpd.get('input_cpd', None)
        
    return sequence

def find_best_rxn_routes(matched_f2products):
    '''Selelct a product with most detected intermediaries, i.e. optimal flux, by detection_score.
    matched_f2products : dict of matched predicted products from n steps.
    returns product of best detection_score
    '''
    pass


def match_features_by_substrate_rxns(
    list_features, dict_predicted_products, 
    mz_ppm=10, rt_tolerance=3000000
    ):
    '''
    Match features to substrate step-wise products using best m/z matching. 
    Do nonredundant featurs first, then add others.

    Args:
        features (list): List of features to match.
        predicted_products (dict): products with rxn steps.
        mz_ppm (float, optional): Mass tolerance in ppm. Defaults to 10.
        rt_tolerance (float, optional): Retention time tolerance. Defaults to 3000000.

    Returns:
        list: updated features with corresponding reaction product.
    '''
    features = deepcopy(list_features)
    for feat in features:
        feat.pop('matched', None)
        feat.pop('related', None)
    _unique, _extra, misfits = get_clean_features(features)
    found, unmatched = {}, []
    product_dict = {}
    for step, predicted_products in dict_predicted_products.items():
        product_dict.update({product['id']: product for product in predicted_products})
        queries = [feat for feat in _unique if feat['id'] not in found]
        # change from best_mz_match_lcms_features to list_match_lcms_features
        # collect all matches then select best via find_best_rxn_routes
        matched = mf.list_match_lcms_features(
            queries, 
            predicted_products, 
            mz_ppm=mz_ppm, rt_tolerance=rt_tolerance
            )
        for feat, prods in matched.items():
            matched[feat] = max(prods, key=lambda x: product_dict[x]['detection_score'])
        found.update(matched)
        
    for feat in features:
        if feat['id'] in found:
            feat['matched'] = product_dict[found[feat['id']]]
            if 'ion_relation' not in feat or not feat['ion_relation']:
                feat['ion_relation'] = ['M0,M+H+']
                feat['neutral_formula_mass'] = feat['mz'] - 1.007276  # mass of a proton
                
        elif feat['id'] in _extra:
            related = _extra[feat['id']]
            feat['related'] = related
            if related in found:
                feat['matched'] = product_dict[found[related]]  
            else:
                unmatched.append(feat)
        else:
            unmatched.append(feat)      # including misfits
    
    return features, unmatched

Khipus were run on the above combinations earlier. 
We may break khipus if we separate metabolites from, say, estrone and estradiol. 

We generate predicted metabolites by substrate, then search against the combined table. 

**DnCl**

In [ ]:

DnCl_pellets_features = json.load(open(featdir +'DnCl_pellets_features.json'))
DnCl_supernatant_features = json.load(open(featdir + 'DnCl_supernatant_features.json'))
len(DnCl_pellets_features), len(DnCl_supernatant_features)

In [ ]:
DnCl_pellets_features[88]

In [ ]:
predicted_3step_products = json.load(open(proddir + 'estradiol_3step_products.json'))

# estradiol_3step_products, cells
features, unmatched = match_features_by_substrate_rxns(
    DnCl_pellets_features, 
    predicted_3step_products)
json.dump(features, open(os.path.join(outdir, 'estradiol_3step_products_pellets_features.json'), 'w'), indent=4)
json.dump(unmatched, open(os.path.join(outdir, 'estradiol_3step_products_pellets_unmatched.json'), 'w'), indent=4)

# estradiol_3step_products, supernatant
features, unmatched = match_features_by_substrate_rxns(
    DnCl_supernatant_features, 
    predicted_3step_products)
json.dump(features, open(os.path.join(outdir, 'estradiol_3step_products_supernatant_features.json'), 'w'), indent=4)
json.dump(unmatched, open(os.path.join(outdir, 'estradiol_3step_products_supernatant_unmatched.json'), 'w'), indent=4)

In [ ]:
predicted_3step_products = json.load(open(proddir + 'estrone_3step_products.json'))

# estrone_3step_products, cells
features, unmatched = match_features_by_substrate_rxns(
    DnCl_pellets_features, 
    predicted_3step_products)
json.dump(features, open(os.path.join(outdir, 'estrone_3step_products_pellets_features.json'), 'w'), indent=4)
json.dump(unmatched, open(os.path.join(outdir, 'estrone_3step_products_pellets_unmatched.json'), 'w'), indent=4)

# estrone_3step_products, supernatant
features, unmatched = match_features_by_substrate_rxns(
    DnCl_supernatant_features, 
    predicted_3step_products)
json.dump(features, open(os.path.join(outdir, 'estrone_3step_products_supernatant_features.json'), 'w'), indent=4)
json.dump(unmatched, open(os.path.join(outdir, 'estrone_3step_products_supernatant_unmatched.json'), 'w'), indent=4)


**DnHz**

In [ ]:
DnHz_supernatant_features = json.load(open(featdir + 'DnHz_supernatant_features.json'))
DnHz_pellets_features = json.load(open(featdir + 'DnHz_pellets_features.json'))
len(DnHz_pellets_features), len(DnHz_supernatant_features)

In [ ]:
predicted_3step_products = json.load(open(proddir + 'dhea_3step_products.json'))

# dhea_3step_products, cells
features, unmatched = match_features_by_substrate_rxns(
    DnHz_pellets_features, 
    predicted_3step_products)
json.dump(features, open(os.path.join(outdir, 'dhea_3step_products_pellets_features.json'), 'w'), indent=4)
json.dump(unmatched, open(os.path.join(outdir, 'dhea_3step_products_pellets_unmatched.json'), 'w'), indent=4)

# dhea_3step_products, supernatant
features, unmatched = match_features_by_substrate_rxns(
    DnHz_supernatant_features, 
    predicted_3step_products)
json.dump(features, open(os.path.join(outdir, 'dhea_3step_products_supernatant_features.json'), 'w'), indent=4)
json.dump(unmatched, open(os.path.join(outdir, 'dhea_3step_products_supernatant_unmatched.json'), 'w'), indent=4)


In [ ]:
predicted_3step_products = json.load(open(proddir + 'pregnenolone_3step_products.json'))

# pregnenolone_3step_products, cells
features, unmatched = match_features_by_substrate_rxns(
    DnHz_pellets_features, 
    predicted_3step_products)
json.dump(features, open(os.path.join(outdir, 'pregnenolone_3step_products_pellets_features.json'), 'w'), indent=4)
json.dump(unmatched, open(os.path.join(outdir, 'pregnenolone_3step_products_pellets_unmatched.json'), 'w'), indent=4)

# pregnenolone_3step_products, supernatant
features, unmatched = match_features_by_substrate_rxns(
    DnHz_supernatant_features, 
    predicted_3step_products)
json.dump(features, open(os.path.join(outdir, 'pregnenolone_3step_products_supernatant_features.json'), 'w'), indent=4)
json.dump(unmatched, open(os.path.join(outdir, 'pregnenolone_3step_products_supernatant_unmatched.json'), 'w'), indent=4)


In [ ]:
predicted_3step_products = json.load(open(proddir + 'progesterone_3step_products.json'))

# progesterone_3step_products, cells
features, unmatched = match_features_by_substrate_rxns(
    DnHz_pellets_features, 
    predicted_3step_products)
json.dump(features, open(os.path.join(outdir, 'progesterone_3step_products_pellets_features.json'), 'w'), indent=4)
json.dump(unmatched, open(os.path.join(outdir, 'progesterone_3step_products_pellets_unmatched.json'), 'w'), indent=4)

# progesterone_3step_products, supernatant
features, unmatched = match_features_by_substrate_rxns(
    DnHz_supernatant_features, 
    predicted_3step_products)
json.dump(features, open(os.path.join(outdir, 'progesterone_3step_products_supernatant_features.json'), 'w'), indent=4)
json.dump(unmatched, open(os.path.join(outdir, 'progesterone_3step_products_supernatant_unmatched.json'), 'w'), indent=4)


## Summary

We have built 3-step reaction networks per substrate. Each run ~20 seconds. 
These networks match almost all candidate features (step wise, 5 ppm). 

Updated features are saved in `*_features.json` and unmatched features also in `_unmatched.json`

The updated feature list has enough details to recover rxn steps.
The unmatched features will be analyzed separately.
Next step to compile features and rxn steps. 